# COMPSCI764 Colab Runner

Orchestration only. Do not execute during Phase 1A. Phase 1B requires separate authorisation, an exact reviewed commit, a reviewed dependency lock, and an explicitly approved candidate profile. All blank inputs below must be resolved before execution. No profile becomes tested until Run A and a fresh-runtime Run B pass and the resulting lock/profile are reviewed and committed. Never push or edit project code from Colab.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/DanielS26/rag-cad-semantic-pairs.git"
COMMIT = ""
BOOTSTRAP_SHA256 = ""
PHASE1B_AUTHORISED = False
ALLOW_NETWORK = False
STAGE = "run-a"
RUN_ID = ""
SOURCE_MANIFEST = ""
RUN_A_MANIFEST = ""
CHECKOUT = Path("/content/rag-cad-semantic-pairs")
WORK = Path("/content/work")
PROFILE_RELATIVE = "configs/environments/colab_candidate.yaml"
LOCK_RELATIVE = "requirements/colab-core.candidate.lock.txt"
STORAGE_RELATIVE = "configs/storage.yaml"
READY = False
STAGE_COMPLETE = False

In [ ]:
import hashlib
import re
import runpy
import urllib.request

if not PHASE1B_AUTHORISED or not RUN_ID or ALLOW_NETWORK is not True:
    raise RuntimeError("Separate Phase 1B and bootstrap-network permission and a new run ID are required")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT) or not re.fullmatch(r"[0-9a-f]{64}", BOOTSTRAP_SHA256):
    raise RuntimeError("Supply the exact reviewed commit and bootstrap file hash")
match = re.fullmatch(r"https://github\.com/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)\.git", REPO_URL)
if match is None:
    raise RuntimeError("Expected a clean GitHub HTTPS URL")
from google.colab import userdata

token = userdata.get("GH_READ_TOKEN")
headers = {"Accept": "application/vnd.github.raw", "Authorization": "Bearer " + token}
request = None
try:
    bootstrap_url = f"https://api.github.com/repos/{match[1]}/{match[2]}/contents/scripts/bootstrap_colab.py?ref={COMMIT}"
    request = urllib.request.Request(bootstrap_url, headers=headers)
    with urllib.request.urlopen(request, timeout=60) as response:
        bootstrap_bytes = response.read()
    if hashlib.sha256(bootstrap_bytes).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError("Bootstrap integrity check failed")
    bootstrap_path = Path("/content/rag_cad_bootstrap.py")
    with bootstrap_path.open("xb") as handle:
        handle.write(bootstrap_bytes)
    bootstrap = runpy.run_path(str(bootstrap_path), run_name="trusted_bootstrap")
    bootstrap["clone_exact"](REPO_URL, COMMIT, CHECKOUT, token, allow_network=ALLOW_NETWORK)
except Exception:
    raise RuntimeError("Bootstrap failed; secret-bearing request details suppressed") from None
finally:
    token = None
    headers.clear()
    request = None

In [ ]:
import subprocess
import sys

READY = False
if not PHASE1B_AUTHORISED or ALLOW_NETWORK is not True:
    raise RuntimeError("Dependency installation requires separately approved network permission")
bootstrap["install_and_test"](CHECKOUT, REPO_URL, COMMIT, LOCK_RELATIVE, allow_network=ALLOW_NETWORK)
WORK.mkdir(parents=True, exist_ok=True)
common = [sys.executable, "-m", "rag_cad.cli", "--repo", str(CHECKOUT)]
gate = ["--profile", str(CHECKOUT / PROFILE_RELATIVE), "--lock", str(CHECKOUT / LOCK_RELATIVE),
        "--commit", COMMIT, "--workspace", str(WORK), "--execute"]
subprocess.run(common + ["environment", "preflight", "--stage", "smoke", *gate,
                        "--output", str(WORK / "preflight.json")], cwd=CHECKOUT, check=True)
READY = True

In [ ]:
if not READY or not PHASE1B_AUTHORISED:
    raise RuntimeError("Verify the trusted checkout and pass preflight before mounting Drive")
bootstrap["verify_checkout"](CHECKOUT, REPO_URL, COMMIT)
from google.colab import drive

drive.mount("/content/drive")
if SOURCE_MANIFEST:
    subprocess.run(common + ["artefact", "restore", "--manifest", SOURCE_MANIFEST,
                             "--destination", str(WORK / "inputs")], cwd=CHECKOUT, check=True)
source = WORK / "inputs" / "upstream"
resources = WORK / "inputs" / "nltk_data"

In [ ]:
STAGE_COMPLETE = False
if not READY or not PHASE1B_AUTHORISED or STAGE not in {"run-a", "run-b"}:
    raise RuntimeError("Only an authorised bounded smoke stage is allowed")
if not source.exists():
    if not ALLOW_NETWORK:
        raise RuntimeError("Restore verified inputs or explicitly authorise named downloads")
    subprocess.run(common + ["upstream", "fetch-cad", *gate,
        "--release", str(CHECKOUT / "configs/upstream/cad_release.yaml"),
        "--source", str(source), "--output", str(WORK / "source_manifest.json"),
        "--allow-network"], cwd=CHECKOUT, check=True)
stage_output = WORK / "outputs" / RUN_ID
stage_command = common + ["smoke", STAGE, *gate,
    "--release", str(CHECKOUT / "configs/upstream/cad_release.yaml"),
    "--source", str(source), "--resources", str(resources),
    "--model-bundle", str(WORK / "inputs" / "jina-bundle")]
if STAGE == "run-a":
    stage_command += ["--output", str(stage_output)]
else:
    if not RUN_A_MANIFEST:
        raise RuntimeError("Run B requires the published Run A manifest")
    stage_command += ["--manifest", RUN_A_MANIFEST, "--destination", str(WORK / "restored_run_a"),
                      "--output", str(stage_output / "run_b.json")]
if ALLOW_NETWORK:
    stage_command.append("--allow-network")
subprocess.run(stage_command, cwd=CHECKOUT, check=True)
STAGE_COMPLETE = True

In [ ]:
if not READY or not PHASE1B_AUTHORISED or not STAGE_COMPLETE:
    raise RuntimeError("Publication requires a completed authorised stage")
subprocess.run(common + ["artefact", "publish", "--source", str(stage_output),
    "--stage", "smoke", "--run-id", RUN_ID,
    "--storage-config", str(CHECKOUT / STORAGE_RELATIVE),
    "--metadata", str(WORK / "preflight.json")], cwd=CHECKOUT, check=True)
print({"commit": COMMIT, "run_id": RUN_ID, "published_and_verified": True})